# Llama 3 Inference Notebook

This notebook demonstrates how to use the Llama 3 model for text generation and chat completion.

## Features
- Text completion from prompts
- Chat completion with conversational formatting
- Multi-turn conversations
- Configurable generation parameters (temperature, top-p, max length)

## Setup for Kaggle

**Before running this notebook in Kaggle:**

### Step 1: Create a Kaggle Dataset

1. **Go to Kaggle Datasets:**
   - Visit https://www.kaggle.com/datasets
   - Click "New Dataset" button

2. **Upload your files from USB:**
   - **If you have safetensors files** (recommended for first upload):
     - Upload all 4 safetensors files:
       - `model-00001-of-00004.safetensors` (~4GB)
       - `model-00002-of-00004.safetensors` (~4GB)
       - `model-00003-of-00004.safetensors` (~4GB)
       - `model-00004-of-00004.safetensors` (~1.1GB)
     - Upload `config.json` (from Hugging Face model)
     - Upload `tokenizer.model`
   
   - **If you already have consolidated.00.pth:**
     - Upload `consolidated.00.pth` (~15GB)
     - Upload `params.json`
     - Upload `tokenizer.model`

3. **Name your dataset:**
   - Give it a name (e.g., "llama-3-8b-instruct")
   - Make it **Private** (recommended) or Public
   - Click "Create"

### Step 2: Add Dataset to Your Notebook

1. **In your Kaggle notebook:**
   - Click "Add data" button (top right)
   - Search for your dataset name
   - Click "Add" next to your dataset
   - The files will be available at `/kaggle/input/your-dataset-name/`

### Step 3: Update Configuration

- In the Configuration cell below, update:
  ```python
  dataset_name = "your-dataset-name"  # Change this!
  ```

### Step 4: Enable GPU

- Go to **Settings** → **Accelerator** → Select **"GPU T4 x2"** or higher
- The model requires GPU for inference

### Step 5: Upload Project Code

**Option A: Upload from USB (if code is on USB):**
- In Kaggle notebook, click "File" → "Upload"
- Upload the entire project folder (or zip it first)
- Extract to `/kaggle/working/` if needed

**Option B: Clone from GitHub (if code is in a repo):**
- Add a code cell at the top:
  ```python
  !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/
  ```

**Option C: Manual upload via Kaggle API:**
- Use Kaggle API to upload files programmatically


## Setup and Imports

First, we need to install dependencies (if needed) and set up the Python path to import the necessary modules.

**Note:** If you haven't uploaded the project code yet, choose one of these methods:

### Method 1: Git Clone (Recommended - Most Dynamic)
If your code is in a GitHub repo, run this cell to clone it:
```python
# Uncomment and update the URL if using git:
# !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/
```

### Method 2: Upload via Notebook
- Click "File" → "Upload" in the notebook
- Select individual files or folders
- Files will be in `/kaggle/working/`

### Method 3: Use Kaggle's Git Integration
- In notebook settings, enable "Git" integration
- Connect your GitHub repo


In [ ]:
# ============================================================================
# OPTION 1: Clone from GitHub (Easiest & Most Dynamic)
# ============================================================================
# If your code is in a GitHub repo, uncomment and run this:
!git clone https://github.com/aalvsz/llama-3-from-scratch.git /kaggle/working/llama3/ || echo "Repository may already exist, continuing..."

# ============================================================================
# OPTION 2: Install Dependencies (if needed)
# ============================================================================
# Uncomment if packages are missing:
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# %pip install -q tiktoken

# ============================================================================
# Setup Python Path
# ============================================================================
import sys
import os
from pathlib import Path

# Detect if running in Kaggle
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # In Kaggle: add /kaggle/working to path
    project_root = Path('/kaggle/working/llama3')
    sys.path.insert(0, str(project_root))
    print("Running in Kaggle environment")
    print(f"Project root: {project_root}")
    
    # Check if src directory exists
    if not (project_root / "src").exists():
        print("\n⚠️  WARNING: src/ directory not found!")
        print("\nTo fix this, choose one of these options:")
        print("\n1. Git Clone (Recommended):")
        print("   !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/")
        print("\n2. Upload Files:")
        print("   - Click 'File' → 'Upload' in the notebook")
        print("   - Upload the 'src' folder and other project files")
        print("\n3. Use Kaggle's Git Integration:")
        print("   - Go to Settings → Enable 'Git'")
        print("   - Connect your GitHub repo")
        raise FileNotFoundError("Project code not found. Please upload or clone the code first.")
    else:
        print("✓ Project code found")
else:
    # Local: add parent directory to path
    project_root = Path().resolve().parent
    sys.path.insert(0, str(project_root))
    print("Running in local environment")
    print(f"Project root: {project_root}")

# Import the Llama inference class
try:
    from src.inference import Llama
    print("✓ Successfully imported Llama class")
except ImportError as e:
    print(f"❌ Failed to import Llama: {e}")
    print("\nMake sure the project code is in the correct location:")
    if IS_KAGGLE:
        print("  - Upload to /kaggle/working/")
        print("  - Or clone from GitHub")
    else:
        print("  - Should be in parent directory")
    raise


fatal: destination path '/kaggle/working/llama3' already exists and is not an empty directory.
Repository may already exist, continuing...
Running in Kaggle environment
Project root: /kaggle/working/llama3
✓ Project code found
✓ Successfully imported Llama class


## Configuration

Configure the paths to your model files. You can provide either:
- **Safetensors files** (`model-00001-of-00004.safetensors`, etc.) - will be converted automatically
- **Consolidated checkpoint** (`consolidated.00.pth`) - ready to use

Also configure generation parameters.


In [2]:
# Detect if running in Kaggle
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # Kaggle paths: model should be in /kaggle/input/your-dataset-name/
    # Update 'your-dataset-name' to match your actual dataset name
    dataset_name = "meta-llama-3-1"  # CHANGE THIS to your dataset name
    model_dir = f"/kaggle/input/{dataset_name}/"  # Directory containing model files
    # Output directory for converted files (if needed)
    output_dir = "/kaggle/working/llama3/checkpoints"  # Where to save consolidated.00.pth if converting
    tokenizer_path = f"/kaggle/input/{dataset_name}/tokenizer.model"  # Path to tokenizer.model file
    print(f"Using Kaggle paths:")
    print(f"  Model directory: {model_dir}")
    print(f"  Output directory: {output_dir}")
    print(f"  Tokenizer: {tokenizer_path}")
else:
    # Local paths
    model_dir = "./checkpoints"  # Directory containing model files (safetensors or consolidated)
    output_dir = "./checkpoints"  # Where to save consolidated.00.pth if converting
    tokenizer_path = "./checkpoints/tokenizer.model"  # Path to tokenizer.model file
    print(f"Using local paths:")
    print(f"  Model directory: {model_dir}")
    print(f"  Output directory: {output_dir}")
    print(f"  Tokenizer: {tokenizer_path}")

# Generation parameters
temperature = 0.6  # Sampling temperature (0.0 = deterministic, higher = more random)
top_p = 0.9       # Top-p (nucleus) sampling parameter
max_seq_len = 128 # Maximum sequence length for the model
max_gen_len = 64  # Maximum number of tokens to generate
max_batch_size = 4  # Maximum batch size for processing multiple prompts


Using Kaggle paths:
  Model directory: /kaggle/input/meta-llama-3-1/
  Output directory: /kaggle/working/llama3/checkpoints
  Tokenizer: /kaggle/input/meta-llama-3-1/tokenizer.model


## Convert Safetensors to Consolidated Format (if needed)

If you have safetensors files instead of consolidated.00.pth, this cell will convert them automatically.


In [ ]:
from pathlib import Path

# Resolve checkpoint directory (consolidated or safetensors are both supported)
model_path = Path(model_dir)
consolidated_path = model_path / "consolidated.00.pth"

if consolidated_path.exists():
    print(f"✓ Found consolidated.00.pth: {consolidated_path}")
    print(f"  Size: {consolidated_path.stat().st_size / 1024**3:.2f} GB")
    ckpt_dir = str(model_path) + '/'  # Ensure trailing slash
    print("\n✓ Ready to use! Skipping conversion.")
else:
    safetensors_files = sorted(model_path.glob("model-*.safetensors"))
    if not safetensors_files:
        print("❌ No model files found!")
        print(f"   Looked in: {model_path}")
        print("\nPlease ensure you have either:")
        print("  1. consolidated.00.pth file, OR")
        print("  2. model-00001-of-00004.safetensors, model-00002-of-00004.safetensors, etc.")
        raise FileNotFoundError("No model files found!")

    print(f"✓ Found {len(safetensors_files)} safetensors file(s)")
    for st_file in safetensors_files:
        size_gb = st_file.stat().st_size / 1024**3
        print(f"  - {st_file.name} ({size_gb:.2f} GB)")

    ckpt_dir = str(model_path) + '/'  # Use safetensors directly (no conversion)
    print("\n✓ Using safetensors directly (no conversion).")
    print("  This avoids huge RAM/disk spikes in Kaggle.")


✓ Found consolidated.00.pth in output directory: /kaggle/working/llama3/checkpoints/consolidated.00.pth
  Size: 14.96 GB

✓ Ready to use! Skipping conversion.


## Verify Files

Quick check that all required files are ready.


In [4]:
# Final verification
# Ensure ckpt_dir ends with / for the inference code
if not ckpt_dir.endswith('/'):
    ckpt_dir = ckpt_dir + '/'

ckpt_path = Path(ckpt_dir.rstrip('/'))  # Remove trailing slash for Path operations
consolidated = ckpt_path / "consolidated.00.pth"
safetensors_files = sorted(ckpt_path.glob("model-*.safetensors"))
tokenizer_file = Path(tokenizer_path)
params_file = ckpt_path / "params.json"

print("Final file check:")
print(f"Checkpoint directory: {ckpt_dir}")

has_weights = consolidated.exists() or bool(safetensors_files)
if consolidated.exists():
    print(f"  ✓ consolidated.00.pth: {consolidated} ({consolidated.stat().st_size / 1024**3:.2f} GB)")
elif safetensors_files:
    print(f"  ✓ safetensors shards: {len(safetensors_files)} files")
    for st in safetensors_files:
        print(f"    - {st.name} ({st.stat().st_size / 1024**3:.2f} GB)")
else:
    print(f"  ✗ weights not found in {ckpt_path}")

if tokenizer_file.exists():
    print(f"  ✓ tokenizer.model: {tokenizer_file}")
else:
    print(f"  ✗ tokenizer.model: {tokenizer_file} - NOT FOUND")

if params_file.exists():
    print(f"  ✓ params.json: {params_file}")
else:
    print(f"  ⚠ params.json: {params_file} - NOT FOUND (optional)")

if not has_weights or not tokenizer_file.exists():
    raise FileNotFoundError("Missing required files! Check paths above.")
else:
    print("\n✓ All required files ready!")


Final file check:
Checkpoint directory: /kaggle/working/llama3/checkpoints/
  ✓ consolidated.00.pth: /kaggle/working/llama3/checkpoints/consolidated.00.pth (14.96 GB)
  ✓ params.json: /kaggle/working/llama3/checkpoints/params.json
  ✓ tokenizer.model: /kaggle/input/meta-llama-3-1/tokenizer.model

✓ All files ready!


## Diagnostic: Check System State

Before loading the model, let's verify the environment is ready.


In [ ]:
# Diagnostic checks before model loading
import torch
import sys
import os

print("=== System Diagnostics ===")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Check for MPS (Metal Performance Shaders) on Apple Silicon
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f"MPS (Metal) available: {mps_available}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"    Total memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"    Allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
        print(f"    Reserved: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")
elif mps_available:
    print("✓ Using MPS (Metal) on Apple Silicon - GPU acceleration enabled")
    print("  Note: MPS memory is managed automatically by macOS")
else:
    print("⚠️  No GPU/MPS available - model will run on CPU (very slow)")

print("\n=== File Check ===")
print(f"Checkpoint dir exists: {os.path.exists(ckpt_dir)}")
print(f"Tokenizer exists: {os.path.exists(tokenizer_path)}")

if os.path.exists(ckpt_dir):
    checkpoint_file = os.path.join(ckpt_dir, "consolidated.00.pth")
    if os.path.exists(checkpoint_file):
        size_gb = os.path.getsize(checkpoint_file) / 1024**3
        print(f"Checkpoint file size: {size_gb:.2f} GB")
    else:
        print(f"⚠️  Checkpoint file not found: {checkpoint_file}")

print("\n=== Ready to load model ===")


## Load Model

Load the pre-trained Llama 3 model. This may take a few minutes as it loads ~8B parameters into GPU memory.

**Note:** If you encounter an OutOfMemoryError, make sure to clear GPU memory first by running the cell below.


In [ ]:
# Clear GPU memory before loading model (important if running cell multiple times)
import torch
import gc

# Delete existing generator instance if it exists (helps free memory in notebooks)
if 'generator' in globals():
    del generator
    print("Deleted existing generator instance")

# Check for MPS (Metal Performance Shaders) on Apple Silicon
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated(0)
    total = torch.cuda.get_device_properties(0).total_memory
    free = total - allocated
    print(f"GPU memory cleared. Free: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB")
    print()
elif mps_available:
    # MPS memory is managed automatically by macOS, but we can still run garbage collection
    gc.collect()
    print("MPS (Metal) memory cleared (managed automatically by macOS)")
    print()

# Detect number of available GPUs
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Detected {num_gpus} GPU(s)" if num_gpus > 0 else f"Using {'MPS (Metal)' if mps_available else 'CPU'}")

print("Loading model...")
print("This may take a few minutes as it loads ~8B parameters into GPU memory...\n")

# Use model parallelism if multiple GPUs are available
# For 2 T4 GPUs, set model_parallel_size=2 to split layers across GPUs
# Start with model_parallel_size=1 (single GPU) to avoid crashes
# You can try model_parallel_size=2 if you have 2+ GPUs and enough memory
# Note: MPS doesn't support multi-GPU parallelism
# Auto-select model parallel size for multi-GPU setups
model_parallel_size = None
if num_gpus >= 2:
    print(f"Note: {num_gpus} GPUs detected. Using model_parallel_size=2 to split layers across GPUs.")
    model_parallel_size = 2
elif mps_available:
    print("Note: MPS (Metal) doesn't support multi-GPU parallelism. Using single device.")
else:
    model_parallel_size = 1
print(f"Using model_parallel_size={model_parallel_size}")
try:
    generator = Llama.build(
        ckpt_dir=ckpt_dir,
        tokenizer_path=tokenizer_path,
        max_seq_len=max_seq_len,
        max_batch_size=max_batch_size,
        model_parallel_size=model_parallel_size
    )
    print("✓ Model loaded successfully!\n")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure GPU memory is cleared (run the cell above)")
    print("2. Try reducing max_seq_len or max_batch_size")
    print("3. If you have 2 GPUs, try model_parallel_size=2")
    raise


GPU memory cleared. Free: 14.74 GB / 14.74 GB

Loading model...
This may take a few minutes as it loads ~8B parameters into GPU memory...



/usr/local/lib/python3.11/dist-packages/torch/__init__.py:1236: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 468.19 MiB is free. Process 20515 has 14.28 GiB memory in use. Of the allocated memory 14.04 GiB is allocated by PyTorch, and 129.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Example 1: Text Completion

Generate text completions from simple prompts. The model will continue the given text.


In [ ]:
prompts = [
    "I believe the meaning of life is",
    "Simply put, the theory of relativity states that ",
]

# Generate completions for all prompts
results = generator.text_completion(
    prompts,
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)

# Print results
for prompt, result in zip(prompts, results):
    print(f"Prompt: {prompt}")
    print(f"Completion: {result['generation']}")
    print("\n" + "=" * 50 + "\n")


## Example 2: Chat Completion

Generate responses in a conversational format with system and user messages.


In [ ]:
# Define a conversation dialog
# Each message has a role (system, user, or assistant) and content
dialogs = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ]
]

# Generate assistant response
chat_results = generator.chat_completion(
    dialogs,
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)

# Print chat results
for dialog, result in zip(dialogs, chat_results):
    print("Dialog:")
    for message in dialog:
        print(f"  {message['role']}: {message['content']}")
    print(f"\nAssistant: {result['generation']['content']}")
    print("\n" + "=" * 50 + "\n")


## Example 3: Multi-turn Conversation

Demonstrate a multi-turn conversation where the model maintains context across multiple exchanges.


In [ ]:
# Start a conversation
conversation = [
    {"role": "user", "content": "Explain quantum computing in simple terms."},
]

# Generate first response
response1 = generator.chat_completion(
    [conversation],
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)[0]

# Add assistant response to conversation
conversation.append(response1["generation"])

# Add follow-up question
conversation.append({"role": "user", "content": "How does it differ from classical computing?"})

# Generate second response
response2 = generator.chat_completion(
    [conversation],
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)[0]

# Print the full conversation
print("Full conversation:")
for message in conversation:
    print(f"  {message['role']}: {message['content']}")
print(f"  {response2['generation']['role']}: {response2['generation']['content']}")


## Custom Prompts

Try your own prompts! Modify the cell below to test different inputs.


In [ ]:
# Custom text completion
custom_prompt = "The future of artificial intelligence will"

result = generator.text_completion(
    [custom_prompt],
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)[0]

print(f"Prompt: {custom_prompt}")
print(f"Completion: {result['generation']}")


## Adjusting Generation Parameters

You can experiment with different generation parameters to control the output:

- **temperature**: Lower values (0.0-0.5) = more deterministic, focused outputs. Higher values (0.7-1.5) = more creative, diverse outputs.
- **top_p**: Nucleus sampling threshold. Lower values (0.5-0.8) = more focused on high-probability tokens. Higher values (0.9-1.0) = broader sampling.
- **max_gen_len**: Maximum number of tokens to generate. Increase for longer outputs.


In [ ]:
# Example with different parameters
test_prompt = "Write a short story about a robot learning to paint."

# More creative (higher temperature)
creative_result = generator.text_completion(
    [test_prompt],
    max_gen_len=100,
    temperature=0.9,  # Higher temperature for more creativity
    top_p=0.95,
)[0]

print("Creative output (temperature=0.9):")
print(creative_result['generation'])
print("\n" + "=" * 50 + "\n")

# More focused (lower temperature)
focused_result = generator.text_completion(
    [test_prompt],
    max_gen_len=100,
    temperature=0.3,  # Lower temperature for more focused output
    top_p=0.8,
)[0]

print("Focused output (temperature=0.3):")
print(focused_result['generation'])
